In [187]:
from langchain_community.document_loaders import CSVLoader
from langchain.docstore.document import Document 
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.text_splitter import RecursiveCharacterTextSplitter

model="mistral-nemo"
llm = ChatOllama(model=model, temperature=0)

In [188]:
# loader = CSVLoader("./t20i_Matches_Data.csv")
# docs = loader.load()
import csv

# columns_to_metadata=["Date", "Channel Name","Content Id", "Content Name","Series Name","Device Type","Device Name","Country","City"]
# columns_to_embed = ["Ad Opportunities", "Ad Requests","Ad Responses","Ad Assets Received","Impressions","Start Quartile","First Quartile","Mid Quartile","Third Quartile","Complete Quartile","Invalid Ad Responses","Ads Not Transcoded","Ads Dropped Due To Duration Mismatch","Ads Overflow","Usable Ads Count","Empty Ad Responses","Users Dropped","Conversion Rate"]

columns_to_embed=["Match Name","Series Name", "Match Date","Team1 Name", "Team1 Runs Scored","Team1 Wickets Fell","Team2 Name","Team2 Runs Scored","Team2 Wickets Fell","Match Venue (Stadium)","Match Venue (City)","Match Venue (Country)","Umpire 1","Umpire 2","Match Referee","Toss Winner","Toss Winner Choice","Match Winner","Match Result Text"]
columns_to_metadata = ["Match Name","Series Name", "Match Date","Team1 Name", "Team1 Runs Scored","Team1 Wickets Fell","Team2 Name","Team2 Runs Scored","Team2 Wickets Fell","Match Venue (Stadium)","Match Venue (City)","Match Venue (Country)","Umpire 1","Umpire 2","Match Referee","Toss Winner","Toss Winner Choice","Match Winner","Match Result Text"]
count = 0
docs = []
# print('before',docs)
with open('csvFiles/t20i_Matches_Data_with_less_data.csv', newline="", encoding='utf-8-sig') as csvfile:
    csv_reader = csv.DictReader(csvfile)
    for i, row in enumerate(csv_reader):
        # if(count==4):
        #     break 
        to_metadata = {col: row[col] for col in columns_to_metadata if col in row}
        values_to_embed = {k: row[k] for k in columns_to_embed if k in row}
        to_embed = "\n".join(f"{k.strip()}: {v.strip()}" for k, v in values_to_embed.items())
        newDoc = Document(page_content=to_embed, metadata=to_metadata)
        docs.append(newDoc)
        # count = count + 1
# print('after',docs[1],docs[2])



In [189]:
print(len(docs))
child_splitter = CharacterTextSplitter(separator = "\n",
                                chunk_size=400, 
                                chunk_overlap=0,
                                length_function=len)
# parent_splitter = CharacterTextSplitter(separator = "\n",
#                                 chunk_size=2000, 
#                                 chunk_overlap=0,
#                                 length_function=len)
# documents = splitter.split_documents(docs)

# print(documents[0],'\n',
# documents[1],'\n',
# documents[2],'\n',
# documents[3],'\n',
# documents[4],'\n',
# documents[5])

2592


In [ ]:
# splitter1 = RecursiveCharacterTextSplitter(
#                                 chunk_size=300, 
#                                 chunk_overlap=0,
#                                 length_function=len)
# documents1 = splitter1.split_documents(docs)

# print(documents1[0],'\n',
# documents1[1],'\n',
# documents1[2],'\n',
# documents1[3],'\n',
# documents1[4],'\n',
# documents1[5])

In [190]:
collection="cricket_match_collection"
# collection="analytics_data_collection"

from langchain.embeddings import HuggingFaceEmbeddings
import chromadb
from chromadb.config import Settings
ollm_embed = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
chroma_client = chromadb.HttpClient(host="localhost", port = 8000, settings=Settings(allow_reset=True, anonymized_telemetry=False))
chroma_client.delete_collection(name=collection)
vectorstore = Chroma(
    # documents=docs,
    client=chroma_client,
    collection_name=collection,
    embedding_function=ollm_embed,
)

# vectorstore = Chroma.from_documents(
#     docs,
#     embedding=ollm_embed,
# )

print(len(vectorstore))

/Users/varun/learn/ai/learn-ai/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


0


In [47]:
vectorstore.similarity_search("2008-02-01")

[Document(metadata={'Channel Name': 'Channel: BEONDTV  - LG  - AU', 'City': 'Sydney', 'Content Id': 'sotg-ep2', 'Content Name': 'Stanley on The Go: Calcutta India', 'Country': 'Australia', 'Date': '08-19-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Series Name': 'Stanley on The Go'}, page_content='Ad Opportunities: 234\nAd Requests: 4\nAd Responses: 4\nAd Assets Received: 11\nImpressions: 7\nStart Quartile: 7\nFirst Quartile: 7\nMid Quartile: 7\nThird Quartile: 7\nComplete Quartile: 7\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 11\nEmpty Ad Responses: 0\nUsers Dropped: 4\nConversion Rate: 63.64%'),
 Document(metadata={'Channel Name': 'Channel: BEONDTV  - LG  - AU', 'City': 'Brisbane', 'Content Id': 'sotg-ep4', 'Content Name': 'Stanley on The Go: Scotland', 'Country': 'Australia', 'Date': '08-19-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Series Name': 'Stanley on The Go'}, page

In [195]:
# from langchain.retrievers import ParentDocumentRetriever
from langchain.retrievers import MultiVectorRetriever
from langchain.storage import InMemoryStore


# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 10, 'filter':{'Date':'07-25-2024'}},
# )
# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 70},
# )
# retriever = vectorstore.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={'score_threshold': 0.5},
# )
# retriever=vectorstore.as_retriever(
#                 search_type="mmr",
#                 search_kwargs={'k': 10, 'fetch_k': 100}
#             )
# retriever=vectorstore.as_retriever(
#                 search_type="mmr",
#                 search_kwargs={'k': 10, 'fetch_k': 1000, 'filter':{'Date':'07-25-2024'},'lambda_mult': 0.75}
#             )
store = InMemoryStore()
id_key = "doc_id"
# retriever = ParentDocumentRetriever(
#     vectorstore=vectorstore,
#     docstore=store,
#     child_splitter=child_splitter,
#     parent_splitter=parent_splitter,
#     search_type='mmr'
# )
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
    # search_type='mmr'
)
import uuid

doc_ids = [str(uuid.uuid4()) for _ in docs]
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# print("doc-ids", doc_ids)
sub_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)

print('sub_docs',sub_docs[0])

retriever.vectorstore.add_documents(sub_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

# retriever.add_documents(docs)


sub_docs page_content='Match Name: Australia Vs India Only T20I
Match Date: 2008-02-01
Team1 Name: India
Team1 Runs Scored: 74.0
Team1 Wickets Fell: 10.0
Team2 Name: Australia
Team2 Runs Scored: 75.0
Team2 Wickets Fell: 1.0
Match Venue (Stadium): Melbourne Cricket Ground
Match Venue (City): Melbourne
Match Venue (Country): Australia
Toss Winner: India
Toss Winner Choice: bat
Match Winner: Australia' metadata={'Match Name': 'Australia Vs India Only T20I', 'Match Date': '2008-02-01', 'Team1 Name': 'India', 'Team1 Runs Scored': '74.0', 'Team1 Wickets Fell': '10.0', 'Team2 Name': 'Australia', 'Team2 Runs Scored': '75.0', 'Team2 Wickets Fell': '1.0', 'Match Venue (Stadium)': 'Melbourne Cricket Ground', 'Match Venue (City)': 'Melbourne', 'Match Venue (Country)': 'Australia', 'Toss Winner': 'India', 'Toss Winner Choice': 'bat', 'Match Winner': 'Australia', 'Match Result Text': 'Australia won by 9 wickets (with 52 balls remaining)', 'doc_id': '3dfff4da-db6e-4507-8108-c2c192bb7bd6'}


In [200]:
retriever.get_relevant_documents("India Vs Australia 5Th T20I match")
# retriever.invoke("India Vs Australia 5Th T20I match")

[Document(metadata={'Match Name': 'India Vs Australia 5Th T20I', 'Match Date': '2023-12-03', 'Team1 Name': 'India', 'Team1 Runs Scored': '160.0', 'Team1 Wickets Fell': '8.0', 'Team2 Name': 'Australia', 'Team2 Runs Scored': '154.0', 'Team2 Wickets Fell': '8.0', 'Match Venue (Stadium)': 'M Chinnaswamy Stadium', 'Match Venue (City)': 'Bengaluru', 'Match Venue (Country)': 'India', 'Toss Winner': 'Australia', 'Toss Winner Choice': 'bowl', 'Match Winner': 'India', 'Match Result Text': 'India won by 6 runs'}, page_content='Match Name: India Vs Australia 5Th T20I\nMatch Date: 2023-12-03\nTeam1 Name: India\nTeam1 Runs Scored: 160.0\nTeam1 Wickets Fell: 8.0\nTeam2 Name: Australia\nTeam2 Runs Scored: 154.0\nTeam2 Wickets Fell: 8.0\nMatch Venue (Stadium): M Chinnaswamy Stadium\nMatch Venue (City): Bengaluru\nMatch Venue (Country): India\nToss Winner: Australia\nToss Winner Choice: bowl\nMatch Winner: India\nMatch Result Text: India won by 6 runs')]

In [203]:
# message = """
# You are an AI language model tasked with answering questions based on a specific context provided below. Make sure your answer is directly derived from the context and avoid adding any information not found within it.
# If the Question contains date in the format yyyy-mm-dd for example 2008-10-01 then consider it as Match Date: 2008-10-01 and provide the answer in relation to that

# Question: {question}

# Context:
# {context}
# """

message = """
"You are an assistant for question-answering tasks. "
    "The context provided has data regarding cricket matches played for years"
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know."
    "\n\n"
Question: {question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm



In [205]:
query="India Vs Australia 5Th T20I"
# query="what is the score of the winning team in the match India Vs Australia 5Th T20I" 
# query="how many wickets did Australia loose in the match India Vs Australia 5Th T20I"
# query="how many matches did India play against Australia in the year 2023" # fail
# query="what is the Match Name that took place on 2008-02-01"
# Retrieve context
# context_docs = retriever.get_relevant_documents(query)
# print(context_docs)
# Join documents into a single string
# context = "".join([doc.page_content for doc in context_docs])

# Debug: Print context to ensure it's correct
# print("Retrieved Context:")
# print(context)

response = rag_chain.invoke(query)

print(response.content)

India won the 5th T20I against Australia by 6 runs. The match was played on 2023-12-03 at M Chinnaswamy Stadium in Bengaluru, India. India scored 160 runs and lost all their wickets (8), while Australia scored 154 runs and also lost all their wickets (8). Australia won the toss and chose to bowl.


In [201]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage
from operator import itemgetter
from langchain.chains.retrieval import  create_retrieval_chain
from langchain.chains.history_aware_retriever import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain

### Contextualize question ###
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)


### Answer question ###
system_prompt = ("""
    We are currently in the year 2024.
                 
    You are an AI language model tasked with answering questions based on a specific context provided below. Make sure your answer is directly derived from the context and avoid adding any information not found within it.
    If the input contains date in the format yyyy-mm-dd for example 2008-10-01 then consider it as Match Date: 2008-10-01 and provide the answer in relation to that

    "{context}"
    """
)
# system_prompt = ("""
#     "You are an assistant for question-answering tasks. "
#     "The context provided has data regarding analytics of multiple channels"
#     "Use the following pieces of retrieved context to answer "
#     "the question. If you don't know the answer, say that you "
#     "don't know."
                 
#     Ensure that if the query is related to channel details, "
#     "such as the channel name, it is included in the response. "

#     Please do not assume anything and provide any fictional information, if the context does not have any related data for the query, you can say "I do not know".
                 
#     You can ask follow up questions by providing relevant data from context to the user, so that they can ask better questions and help you in finding appropriate answer.
                 

#     "\n\n"
#     Context: 
#     "{context}"
# """
# )
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


# def debug_retrieval_process(input_query):
#     # Invoke the chain to get the retrieved documents
#     retrieved_docs = history_aware_retriever.invoke({"input": input_query})
    
#     # Log the reformulated question (this would be the same as input_query if not reformulated)
#     reformulated_question = input_query
#     print("Reformulated Question:", reformulated_question)
    
#     # Log the retrieved documents
#     for i, doc in enumerate(retrieved_docs):
#         print(f"\nRetrieved Document {i+1} Metadata:")
#         for key, value in doc.metadata.items():
#             print(f"{key}: {value}")
#         print(f"Retrieved Document {i+1} Content:\n{doc.page_content}")

#     return reformulated_question, retrieved_docs

# # Use the function to debug the retrieval process
# input_query = "in the year 2023 which team has won most matches"
# reformulated_question, retrieved_docs = debug_retrieval_process(input_query)

# # Now use these results in the final question-answering step
# final_answer = question_answer_chain.invoke({
#     "input": reformulated_question,
#     "context": retrieved_docs,
#     "chat_history": []
# })
# print("\nFinal Answer:", final_answer)

### Statefully manage chat history ###
store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
# response = rag_chain.invoke("among all the matches that have happened, when did the first australia vs india match happen")
# config = {"configurable": {"session_id": "first"}}



In [207]:

conversational_rag_chain.invoke(
    # {"input":"what is the conversation rate for the channel Channel: BEONDTV  Platform: LG  Delivery Region: AU"}, # correct based on given context
    # {"input":"in the year 2023 which team has won most matches"},
    # {"input":"India Vs Australia 5Th T20I"},
    {"input":"how many wickets did the winning team loose in the match India Vs Australia 5Th T20I"},
    # {"input":"what's the score of the winning team in that match India Vs Australia 5Th T20I"},
    # {"input":"ok, how about a channel name with impressions less than 20 and greater than 5"},
    # {"input":"what is the channel name"},
    # {"input": "what's the score of Australia in that match"},
    # {"input": "can you provide all the available details of that match"},
    # {"input": "here runs scored says 154.0 why did you say 245/6?"},
    # {"input":"so what's the score of Australia in the match India Vs Australia 5Th T20I"},
    # {"input":"where did the match took place"},
    # {"input": "how many wickets did india loose in that match"},
    # {"input": "what about the other team?"},
    # {"input": "and how much did it score"},
    # {"input": "how much did australia score?"},
    config={
        "configurable": {"session_id": "2"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'The provided context does not include information about a match between India and Australia, specifically for the 5th T20I. Therefore, I cannot determine how many wickets the winning team lost in that particular match based on the given context.'

In [147]:
print(get_session_history('1'))

Human: who lost in the match India Vs Australia 5Th T20I
AI: According to the context, Team2 Name is Australia and they scored 154.0 runs with 8.0 wickets fell. Since their score was less than that of Team1 (India), who scored 160.0 runs with 8.0 wickets fell, it can be inferred that Australia lost in the match India Vs Australia 5Th T20I.
Human: how many wickets did the winning team loose in the match India Vs Australia 5Th T20I
AI: The context does not provide information about a match named "India Vs Australia 5Th T20I". The provided context only contains details for two matches: 

1. Australia Vs India 2Nd T20I (2016-01-29)
2. India Vs Australia 1St T20I (2017-10-07)

There is no information about a match named "India Vs Australia 5Th T20I".


In [11]:
r = "[Document(metadata={'Ad Assets Received': '0', 'Ad Opportunities': '11', 'Ad Requests': '1', 'Ad Responses': '1', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: Klowd TV  Delivery Region: US', 'City': 'Austin', 'Complete Quartile': '0', 'Content Id': 'cl_ep_103_fast', 'Content Name': 'Carlos  Lisa', 'Conversion Rate': '0.00%', 'Country': 'United States', 'Date': '07-24-2024', 'Device Name': 'Others', 'Device Type': 'Others', 'Empty Ad Responses': '0', 'First Quartile': '0', 'Impressions': '0', 'Invalid Ad Responses': '0', 'Mid Quartile': '0', 'Series Name': '', 'Start Quartile': '0', 'Third Quartile': '0', 'Usable Ads Count': '0', 'Users Dropped': '0'}, page_content='Date: 07-24-2024\nChannel Name: Channel: BEONDTV  Platform: Klowd TV  Delivery Region: US\nContent Id: cl_ep_103_fast\nContent Name: Carlos  Lisa\nSeries Name: \nDevice Type: Others\nDevice Name: Others\nCountry: United States\nCity: Austin\nAd Opportunities: 11\nAd Requests: 1\nAd Responses: 1\nAd Assets Received: 0\nImpressions: 0\nStart Quartile: 0\nFirst Quartile: 0\nMid Quartile: 0\nThird Quartile: 0\nComplete Quartile: 0\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 0\nEmpty Ad Responses: 0\nUsers Dropped: 0\nConversion Rate: 0.00%'),Document(metadata={'Ad Assets Received': '6', 'Ad Opportunities': '349', 'Ad Requests': '6', 'Ad Responses': '6', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Sydney', 'Complete Quartile': '6', 'Content Id': 'author-ep1', 'Content Name': 'BEONDTV Author Series', 'Conversion Rate': '100.00%', 'Country': 'Australia', 'Date': '07-25-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '6', 'Impressions': '6', 'Invalid Ad Responses': '0', 'Mid Quartile': '6', 'Series Name': 'BEONDTV Author Series', 'Start Quartile': '6', 'Third Quartile': '6', 'Usable Ads Count': '6', 'Users Dropped': '0'}, page_content='Date: 07-25-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: author-ep1\nContent Name: BEONDTV Author Series\nSeries Name: BEONDTV Author Series\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Sydney\nAd Opportunities: 349\nAd Requests: 6\nAd Responses: 6\nAd Assets Received: 6\nImpressions: 6\nStart Quartile: 6\nFirst Quartile: 6\nMid Quartile: 6\nThird Quartile: 6\nComplete Quartile: 6\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 6\nEmpty Ad Responses: 0\nUsers Dropped: 0\nConversion Rate: 100.00%'),Document(metadata={'Ad Assets Received': '7', 'Ad Opportunities': '307', 'Ad Requests': '6', 'Ad Responses': '6', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Sydney', 'Complete Quartile': '7', 'Content Id': 'cl_ep_103_fast', 'Content Name': 'Carlos  Lisa', 'Conversion Rate': '100.00%', 'Country': 'Australia', 'Date': '07-23-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '7', 'Impressions': '7', 'Invalid Ad Responses': '0', 'Mid Quartile': '7', 'Series Name': '', 'Start Quartile': '7', 'Third Quartile': '7', 'Usable Ads Count': '7', 'Users Dropped': '0'}, page_content='Date: 07-23-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: cl_ep_103_fast\nContent Name: Carlos  Lisa\nSeries Name: \nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Sydney\nAd Opportunities: 307\nAd Requests: 6\nAd Responses: 6\nAd Assets Received: 7\nImpressions: 7\nStart Quartile: 7\nFirst Quartile: 7\nMid Quartile: 7\nThird Quartile: 7\nComplete Quartile: 7\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 7\nEmpty Ad Responses: 0\nUsers Dropped: 0\nConversion Rate: 100.00%'),Document(metadata={'Ad Assets Received': '5', 'Ad Opportunities': '349', 'Ad Requests': '2', 'Ad Responses': '2', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Adelaide', 'Complete Quartile': '2', 'Content Id': 'author-ep1', 'Content Name': 'BEONDTV Author Series', 'Conversion Rate': '40.00%', 'Country': 'Australia', 'Date': '07-25-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '2', 'Impressions': '2', 'Invalid Ad Responses': '0', 'Mid Quartile': '2', 'Series Name': 'BEONDTV Author Series', 'Start Quartile': '2', 'Third Quartile': '2', 'Usable Ads Count': '5', 'Users Dropped': '3'}, page_content='Date: 07-25-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: author-ep1\nContent Name: BEONDTV Author Series\nSeries Name: BEONDTV Author Series\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Adelaide\nAd Opportunities: 349\nAd Requests: 2\nAd Responses: 2\nAd Assets Received: 5\nImpressions: 2\nStart Quartile: 2\nFirst Quartile: 2\nMid Quartile: 2\nThird Quartile: 2\nComplete Quartile: 2\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 5\nEmpty Ad Responses: 0\nUsers Dropped: 3\nConversion Rate: 40.00%'),Document(metadata={'Ad Assets Received': '1', 'Ad Opportunities': '349', 'Ad Requests': '1', 'Ad Responses': '1', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Melbourne', 'Complete Quartile': '1', 'Content Id': 'author-ep1', 'Content Name': 'BEONDTV Author Series', 'Conversion Rate': '100.00%', 'Country': 'Australia', 'Date': '07-25-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '1', 'Impressions': '1', 'Invalid Ad Responses': '0', 'Mid Quartile': '1', 'Series Name': 'BEONDTV Author Series', 'Start Quartile': '1', 'Third Quartile': '1', 'Usable Ads Count': '1', 'Users Dropped': '0'}, page_content='Date: 07-25-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: author-ep1\nContent Name: BEONDTV Author Series\nSeries Name: BEONDTV Author Series\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Melbourne\nAd Opportunities: 349\nAd Requests: 1\nAd Responses: 1\nAd Assets Received: 1\nImpressions: 1\nStart Quartile: 1\nFirst Quartile: 1\nMid Quartile: 1\nThird Quartile: 1\nComplete Quartile: 1\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 1\nEmpty Ad Responses: 0\nUsers Dropped: 0\nConversion Rate: 100.00%'),Document(metadata={'Ad Assets Received': '2', 'Ad Opportunities': '349', 'Ad Requests': '3', 'Ad Responses': '3', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Perth', 'Complete Quartile': '1', 'Content Id': 'author-ep1', 'Content Name': 'BEONDTV Author Series', 'Conversion Rate': '50.00%', 'Country': 'Australia', 'Date': '07-25-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '1', 'Impressions': '1', 'Invalid Ad Responses': '0', 'Mid Quartile': '1', 'Series Name': 'BEONDTV Author Series', 'Start Quartile': '1', 'Third Quartile': '1', 'Usable Ads Count': '2', 'Users Dropped': '1'}, page_content='Date: 07-25-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: author-ep1\nContent Name: BEONDTV Author Series\nSeries Name: BEONDTV Author Series\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Perth\nAd Opportunities: 349\nAd Requests: 3\nAd Responses: 3\nAd Assets Received: 2\nImpressions: 1\nStart Quartile: 1\nFirst Quartile: 1\nMid Quartile: 1\nThird Quartile: 1\nComplete Quartile: 1\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 2\nEmpty Ad Responses: 0\nUsers Dropped: 1\nConversion Rate: 50.00%'),Document(metadata={'Ad Assets Received': '7', 'Ad Opportunities': '307', 'Ad Requests': '4', 'Ad Responses': '4', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Melbourne', 'Complete Quartile': '3', 'Content Id': 'cl_ep_103_fast', 'Content Name': 'Carlos  Lisa', 'Conversion Rate': '57.14%', 'Country': 'Australia', 'Date': '07-23-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '4', 'Impressions': '4', 'Invalid Ad Responses': '0', 'Mid Quartile': '3', 'Series Name': '', 'Start Quartile': '4', 'Third Quartile': '3', 'Usable Ads Count': '7', 'Users Dropped': '3'}, page_content='Date: 07-23-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: cl_ep_103_fast\nContent Name: Carlos  Lisa\nSeries Name: \nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Melbourne\nAd Opportunities: 307\nAd Requests: 4\nAd Responses: 4\nAd Assets Received: 7\nImpressions: 4\nStart Quartile: 4\nFirst Quartile: 4\nMid Quartile: 3\nThird Quartile: 3\nComplete Quartile: 3\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 7\nEmpty Ad Responses: 0\nUsers Dropped: 3\nConversion Rate: 57.14%'),Document(metadata={'Ad Assets Received': '7', 'Ad Opportunities': '410', 'Ad Requests': '2', 'Ad Responses': '2', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Melbourne', 'Complete Quartile': '5', 'Content Id': 'cwc-s4ep1', 'Content Name': 'Conversations With Carroll', 'Conversion Rate': '71.43%', 'Country': 'Australia', 'Date': '08-03-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '5', 'Impressions': '5', 'Invalid Ad Responses': '0', 'Mid Quartile': '5', 'Series Name': 'Conversations With Carroll', 'Start Quartile': '5', 'Third Quartile': '5', 'Usable Ads Count': '7', 'Users Dropped': '2'}, page_content='Date: 08-03-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: cwc-s4ep1\nContent Name: Conversations With Carroll\nSeries Name: Conversations With Carroll\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Melbourne\nAd Opportunities: 410\nAd Requests: 2\nAd Responses: 2\nAd Assets Received: 7\nImpressions: 5\nStart Quartile: 5\nFirst Quartile: 5\nMid Quartile: 5\nThird Quartile: 5\nComplete Quartile: 5\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 7\nEmpty Ad Responses: 0\nUsers Dropped: 2\nConversion Rate: 71.43%'),Document(metadata={'Ad Assets Received': '7', 'Ad Opportunities': '387', 'Ad Requests': '4', 'Ad Responses': '4', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Sydney', 'Complete Quartile': '3', 'Content Id': 'author-ep1', 'Content Name': 'BEONDTV Author Series', 'Conversion Rate': '57.14%', 'Country': 'Australia', 'Date': '07-26-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '4', 'Impressions': '4', 'Invalid Ad Responses': '0', 'Mid Quartile': '4', 'Series Name': 'BEONDTV Author Series', 'Start Quartile': '4', 'Third Quartile': '3', 'Usable Ads Count': '7', 'Users Dropped': '3'}, page_content='Date: 07-26-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: author-ep1\nContent Name: BEONDTV Author Series\nSeries Name: BEONDTV Author Series\nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Sydney\nAd Opportunities: 387\nAd Requests: 4\nAd Responses: 4\nAd Assets Received: 7\nImpressions: 4\nStart Quartile: 4\nFirst Quartile: 4\nMid Quartile: 4\nThird Quartile: 3\nComplete Quartile: 3\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 7\nEmpty Ad Responses: 0\nUsers Dropped: 3\nConversion Rate: 57.14%'),Document(metadata={'Ad Assets Received': '10', 'Ad Opportunities': '323', 'Ad Requests': '5', 'Ad Responses': '5', 'Ads Dropped Due To Duration Mismatch': '0', 'Ads Not Transcoded': '0', 'Ads Overflow': '0', 'Channel Name': 'Channel: BEONDTV  Platform: LG  Delivery Region: AU', 'City': 'Sydney', 'Complete Quartile': '4', 'Content Id': 'cl_ep_103_fast', 'Content Name': 'Carlos  Lisa', 'Conversion Rate': '40.00%', 'Country': 'Australia', 'Date': '07-24-2024', 'Device Name': 'LG TV', 'Device Type': 'Smart TV', 'Empty Ad Responses': '0', 'First Quartile': '4', 'Impressions': '4', 'Invalid Ad Responses': '0', 'Mid Quartile': '4', 'Series Name': '', 'Start Quartile': '4', 'Third Quartile': '4', 'Usable Ads Count': '10', 'Users Dropped': '6'}, page_content='Date: 07-24-2024\nChannel Name: Channel: BEONDTV  Platform: LG  Delivery Region: AU\nContent Id: cl_ep_103_fast\nContent Name: Carlos  Lisa\nSeries Name: \nDevice Type: Smart TV\nDevice Name: LG TV\nCountry: Australia\nCity: Sydney\nAd Opportunities: 323\nAd Requests: 5\nAd Responses: 5\nAd Assets Received: 10\nImpressions: 4\nStart Quartile: 4\nFirst Quartile: 4\nMid Quartile: 4\nThird Quartile: 4\nComplete Quartile: 4\nInvalid Ad Responses: 0\nAds Not Transcoded: 0\nAds Dropped Due To Duration Mismatch: 0\nAds Overflow: 0\nUsable Ads Count: 10\nEmpty Ad Responses: 0\nUsers Dropped: 6\nConversion Rate: 40.00%')]"
print(len(r))

13885
